# Три полносвязные сети для определения тренда

**Задача:** по десяти последовательным ценам закрытия определить общий тренд следующих трёх периодов.

| Класс задания | Направление | Индекс внутри PyTorch |
|---:|---|---:|
| −1 | Нисходящий | 0 |
| 0 | Боковой | 1 |
| 1 | Восходящий | 2 |

Создаём три разные сети по образцу `task3.py`. Используем файл `rosneft_close.csv` со столбцами `NUMBER;CLOSE`. Файл должен находиться рядом с ноутбуком; в Colab его нужно загрузить в текущую папку.

Нейронная сеть — функция с настраиваемыми весами и смещениями. На обучении она получает десять прошлых цен, выдаёт три оценки классов и меняет параметры, уменьшая ошибку относительно известного ответа. Будущие цены используются только для подготовки правильных ответов и не поступают на вход сети.

Предсказываем один класс общего тренда, а не три будущие цены.

## 1. Библиотеки и параметры

`pandas` читает CSV и показывает таблицы, `torch` работает с тензорами и обучает сети, `nn` содержит слои и функцию ошибки.

`TREND_THRESHOLD = 1.5` сохраняем из образца: это абсолютный порог наклона в единицах цены за период, а не фиксированный процент. Условие задания не задаёт порог бокового тренда.

`EPOCHS = 2000` — количество проходов обучения. `manual_seed` фиксирует генератор случайных чисел, чтобы начальные веса были воспроизводимыми при одинаковых условиях запуска.

In [1]:
import pandas as pd
import torch
from torch import nn


TREND_THRESHOLD = 1.5  # Единицы цены за один период.
EPOCHS = 2000

torch.manual_seed(42)

## 2. Чтение цен

Указываем разделитель `;`, выбираем только `CLOSE` и превращаем столбец в список. Порядок записей сохраняется; он должен соответствовать последовательным периодам. `NUMBER` на вход сети не подаётся.

In [2]:
close_prices = pd.read_csv("rosneft_close.csv", sep=";")["CLOSE"].tolist()

print("Количество цен:", len(close_prices))
print("Первые десять цен:", close_prices[:10])

Количество цен: 675
Первые десять цен: [527.4, 508.0, 516.9, 510.45, 517.25, 524.15, 531.85, 530.55, 532.25, 532.45]


## 3. Определение правильного класса

По трём будущим ценам подбираем прямую $p(t)=kt+b$ методом наименьших квадратов. Вычисляем её наклон:

$$
k=\frac{\sum_{i=1}^{3}(t_i-\bar t)(p_i-\bar p)}
{\sum_{i=1}^{3}(t_i-\bar t)^2}, \qquad t=[1,2,3].
$$

`sum` складывает значения, `zip(periods, prices)` соединяет номер периода с его ценой. Числитель связывает изменения времени и цены, знаменатель учитывает расположение периодов.

- $k > 1.5$: восходящий тренд, класс `1`.
- $k < -1.5$: нисходящий тренд, класс `−1`.
- Иначе: боковой тренд, класс `0`.

Для трёх равноотстоящих точек формула упрощается до $k=(p_3-p_1)/2$. Поэтому правило определяет общее направление по наклону, а не требует роста или падения на каждом отдельном шаге.

In [3]:
def calculate_slope(prices):
    periods = [1, 2, 3]
    mean_period = sum(periods) / 3
    mean_price = sum(prices) / 3

    numerator = sum(
        (period - mean_period) * (price - mean_price)
        for period, price in zip(periods, prices)
    )
    denominator = sum((period - mean_period) ** 2 for period in periods)
    return numerator / denominator


def classify_trend(future_prices):
    slope = calculate_slope(future_prices)
    if slope > TREND_THRESHOLD:
        return 1
    elif slope < -TREND_THRESHOLD:
        return -1
    return 0

## 4. Формирование входов и ответов

Каждое окно содержит 13 цен: первые десять — вход, следующие три — для вычисления ответа.

| Начало окна | Вход | Цены для ответа |
|---:|---|---|
| 0 | Периоды 1–10 | Периоды 11–13 |
| 1 | Периоды 2–11 | Периоды 12–14 |
| 2 | Периоды 3–12 | Периоды 13–15 |

Из $N$ цен получается $N-13+1=N-12$ примеров. Правая граница среза Python не включается.

Из каждой входной цены вычитаем последнюю цену **этого же входного окна**, как в образце. Например, `[100, 102, 101]` превращается в `[-1, 1, 0]`. Это убирает абсолютный уровень цены, сохраняя движение внутри окна. Масштаб колебаний при этом сохраняется.

Прибавление `1` переводит классы `−1, 0, 1` в индексы `0, 1, 2`. `float32` используется для входных чисел, `long` — для целочисленных индексов классов. При $M$ примерах входной тензор имеет форму `(M, 10)`, а ответы — `(M,)`.

In [4]:
def make_dataset(prices):
    inputs = []
    classes = []

    for start in range(len(prices) - 12):
        history = prices[start:start + 10]
        future = prices[start + 10:start + 13]

        # Десять цен относительно последней цены входного окна.
        inputs.append([price - history[-1] for price in history])
        # Индексы классов: 0 — вниз, 1 — боковой, 2 — вверх.
        classes.append(classify_trend(future) + 1)

    return (
        torch.tensor(inputs, dtype=torch.float32),
        torch.tensor(classes, dtype=torch.long),
    )

## 5. Обучающая и тестовая выборки

Первые 80% цен используются для обучения, последние 20% — для проверки. Сначала разделяем ряд и только потом формируем окна: цены из тестового участка не используются в обучающих примерах.

Соседние окна внутри одной выборки могут пересекаться. Между обучением и тестом пересечений нет. Окна, пересекающие границу участков, не используем.

In [5]:
border = int(len(close_prices) * 0.8)
train_inputs, train_classes = make_dataset(close_prices[:border])
test_inputs, test_classes = make_dataset(close_prices[border:])

print("Обучающие входы:", tuple(train_inputs.shape))
print("Обучающие ответы:", tuple(train_classes.shape))
print("Тестовые входы:", tuple(test_inputs.shape))
print("Тестовые ответы:", tuple(test_classes.shape))

Обучающие входы: (528, 10)
Обучающие ответы: (528,)
Тестовые входы: (123, 10)
Тестовые ответы: (123,)


## 6. Три разные сети

В полносвязном слое каждый нейрон получает все значения предыдущего слоя и вычисляет $z=\sum_j w_jx_j+b$. Веса $w_j$ и смещение $b$ меняются при обучении.

`nn.Linear(a, b)` преобразует `a` значений в `b`. `nn.Sequential` выполняет слои последовательно.

| Сеть | Размеры слоёв | Активация |
|---|---|---|
| 1 | 10 → 24 → 3 | Tanh |
| 2 | 10 → 32 → 3 | ReLU |
| 3 | 10 → 24 → 12 → 3 | Tanh |

`Tanh` преобразует числа в диапазон от −1 до 1. `ReLU(z)=max(0,z)` заменяет отрицательные числа нулями. Эти функции добавляют нелинейность: без них несколько линейных слоёв можно было бы объединить в один.

Скрытые размеры `24`, `32` и `12` — выбранное число нейронов, не число периодов. Последние три выхода всегда соответствуют индексам классов: `0` — вниз, `1` — боковой, `2` — вверх. Выходы являются оценками, а не вероятностями.

In [6]:
networks = {
    "Сеть 1: 10 → 24 → 3, Tanh": nn.Sequential(
        nn.Linear(10, 24),
        nn.Tanh(),
        nn.Linear(24, 3),
    ),
    "Сеть 2: 10 → 32 → 3, ReLU": nn.Sequential(
        nn.Linear(10, 32),
        nn.ReLU(),
        nn.Linear(32, 3),
    ),
    "Сеть 3: 10 → 24 → 12 → 3, Tanh": nn.Sequential(
        nn.Linear(10, 24),
        nn.Tanh(),
        nn.Linear(24, 12),
        nn.Tanh(),
        nn.Linear(12, 3),
    ),
}

## 7. Обучение

Каждая сеть обучается независимо на всей обучающей выборке, содержащей разные классы.

`CrossEntropyLoss` сравнивает три оценки сети с индексом правильного класса. Дополнительный `Softmax` перед ней не нужен: необходимое преобразование предусмотрено внутри функции ошибки. Для одного примера смысл ошибки — $-\ln p_{\text{правильного класса}}$.

`Adam` обновляет параметры, полученные через `network.parameters()`. `lr=0.01` задаёт скорость обучения.

В каждой эпохе:

1. `zero_grad()` очищает градиенты предыдущего шага.
2. `network(train_inputs)` вычисляет оценки всех обучающих примеров, получая тензор `(M, 3)`.
3. Функция ошибки сравнивает оценки с правильными классами.
4. `backward()` вычисляет градиенты: как параметры влияют на ошибку. Параметры пока не меняются.
5. `step()` изменяет веса и смещения.

`train()` включает режим обучения. Словарь `train_losses` сохраняет последнюю ошибку каждой сети для последующего вывода.

In [7]:
train_losses = {}

for name, network in networks.items():
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(network.parameters(), lr=0.01)

    network.train()
    for epoch in range(EPOCHS):
        optimizer.zero_grad()
        outputs = network(train_inputs)
        loss = loss_function(outputs, train_classes)
        loss.backward()
        optimizer.step()

    train_losses[name] = loss.item()

## 8. Проверка точности

`eval()` включает режим проверки, `no_grad()` отключает вычисление градиентов. Параметры больше не обновляются.

`argmax(dim=1)` выбирает индекс максимальной оценки в каждой строке. Например, `[2.4, -0.7, 0.8]` даёт индекс `0`, то есть нисходящий тренд.

Сравнение с правильными классами возвращает `True` и `False`. `.float()` преобразует их в `1.0` и `0.0`, `.mean()` вычисляет долю верных ответов, `.item()` извлекает обычное число Python. При форматировании результатов переводим долю в проценты.

In [8]:
def get_accuracy(network, inputs, classes):
    network.eval()
    with torch.no_grad():
        predictions = network(inputs).argmax(dim=1)
    return (predictions == classes).float().mean().item()

results = []
for name, network in networks.items():
    results.append({
        "Сеть": name,
        "Ошибка обучения": train_losses[name],
        "Точность на обучении, %": get_accuracy(network, train_inputs, train_classes) * 100,
        "Точность на тесте, %": get_accuracy(network, test_inputs, test_classes) * 100,
    })

pd.DataFrame(results).round(2)

,Сеть,Ошибка обучения,"Точность на обучении, %","Точность на тесте, %"
0,"Сеть 1: 10 → 24 → 3, Tanh",0.47,79.92,29.27
1,"Сеть 2: 10 → 32 → 3, ReLU",0.45,82.95,31.71
2,"Сеть 3: 10 → 24 → 12 → 3, Tanh",0.16,93.37,25.20


Высокая точность на обучении при низкой точности на тесте указывает на переобучение: сеть хорошо подстроилась под обучающие примеры, но плохо переносит это на более поздний участок ряда. Код реализует задачу классификации, однако качество прогноза нужно оценивать по тестовым результатам.